In [ ]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

In [ ]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V02/UCSD-P07-V02-CE2_09.53.27_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V02/UCSD-P07-V02-CE2_09.53.27_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

In [ ]:
seg_type = 'nifti'

seg_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V02/UCSD-P07-V02-CE2_MC_VOI.nii.gz'
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

# Testing the motion compensation, right now is hard coded
seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Figure 1 Display the motion compensation from B mode and CEUS


Figure 1: Axial Plane of the 3D contrast enhanced ultrasound and B mode in 5 consecutive frames. Top row: no motion compensation. Bottom row: motion compensated data. The red dash line represents the boarder of the region of interests


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import binary_erosion

def get_mask_boundary(mask_slice):
    if mask_slice.max() == 0:
        return np.zeros_like(mask_slice, dtype=bool)
    eroded = binary_erosion(mask_slice)
    return mask_slice.astype(bool) & ~eroded

def get_voi_center(mask_3d):
    coords = np.where(mask_3d > 0)
    if len(coords[0]) == 0:
        return None, None, None
    return (int(np.mean(coords[0])),   # lateral  X
            int(np.mean(coords[1])),   # depth    Y
            int(np.mean(coords[2])))   # elevation Z
def enhance_bmode_noise(image_slice, p_low_percentile=15.0, p_high_percentile=98.5):
    non_zero = image_slice[image_slice != 0]
    p_low = np.percentile(non_zero, p_low_percentile)
    p_high = np.percentile(non_zero, p_high_percentile)
    clipped = np.clip(image_slice, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)

def apply_clahe(img_u8, clip=2.0, grid=8):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    return clahe.apply(img_u8)

In [ ]:
from scipy.ndimage import shift as ndshift

# --- voxel spacing (mm). extras_dict stores it as (z, y, x) ---
# volume axes are (X=lateral, Y=depth, Z=elevation, T)
sz, sy, sx = image_data.pixdim   # z, y, x  in mm

nx, ny, nz, num_frames = image_data.pixel_data.shape

show_frames = [20, 21, 22, 23, 24]
ticks = [0, 20, 40, 60, 80]

mask = seg_data.seg_mask                                  # fixed reference VOI
lat_c, dep_c, ele_c = get_voi_center(mask)
bnd = get_mask_boundary(np.transpose(mask[:, :, ele_c]))  # same contour everywhere
roi_top_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).min() * sy
roi_bot_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).max() * sy

mc = seg_data.motion_compensation
ext_axial = [0, nx * sx, ny * sy, 0]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for col, frame in enumerate(show_frames):
    Vol = bmode_image_data.pixel_data[:, :, :, frame]

    # --- row 0: NON-COMPENSATED (raw volume) ---
    raw = np.transpose(Vol[:, :, ele_c]).astype(np.float64)
    img = enhance_bmode_noise(raw, p_low_percentile=5.0, p_high_percentile=99.5) 
    axes[0, col].imshow(img, cmap='gray', extent=ext_axial, aspect='equal')
    axes[0, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[0, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

    # --- row 1: COMPENSATED (volume registered back) ---
    dx, dy, dz = mc.get_translation(frame)
    Vol_mc = ndshift(Vol, shift=[-dx, -dy, -dz], order=1, cval=0)
    raw_mc = np.transpose(Vol_mc[:, :, ele_c]).astype(np.float64)
    img_mc = enhance_bmode_noise(raw_mc, p_low_percentile=5.0, p_high_percentile=99.5)
    # img_mc = apply_clahe(img_mc.astype(np.uint8), clip=3, grid=8) 
    axes[1, col].imshow(img_mc, cmap='gray', extent=ext_axial, aspect='equal')
    axes[1, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[1, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

# Apply tick marks to all axes
for ax in axes.flat:
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis='both', labelsize=16)
    

# Only show tick labels on the bottom row (x) and left column (y)
for col in range(5):
    axes[0, col].set_xticklabels([])          # hide x labels on top row
    if col != 0:
        axes[0, col].set_yticklabels([])      # hide y labels except leftmost
        axes[1, col].set_yticklabels([])

plt.tight_layout()
plt.show()

In [ ]:
# CEUS images enhance with noise reduction from first frame
def compute_ceus_noise_floor(first_frame) -> float:
    """
    Compute the noise floor (p_low scalar) from pre-contrast frames of a 4D CEUS scan.
    Returns the noise floor value as a float.
    """
    pixel_data = first_frame

    ref_frames = pixel_data
    ref_nonzero = ref_frames[ref_frames != 0]
    if ref_nonzero.size == 0:
        raise ValueError("Pre-contrast reference frames contain no non-zero values.")

    noise_mean = np.mean(ref_nonzero)
    noise_std = np.std(ref_nonzero)
    return float(noise_mean+noise_std)  # noise floor is mean + std

def enhance_ceus(slice, p_low, p_high_percentile=99.5):
    """
    Enhance a single CEUS image slice using noise floor and high percentile.
    Returns the enhanced image slice.
    """
    non_zero = slice[slice != 0]
    if non_zero.size == 0:
        return np.zeros_like(slice, dtype=np.uint8)

    p_high = np.percentile(non_zero, p_high_percentile)
    clipped = np.clip(slice, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)

In [ ]:
show_frames = [20, 21, 22, 23, 24]

mask = seg_data.seg_mask                                  # fixed reference VOI
lat_c, dep_c, ele_c = get_voi_center(mask)
bnd = get_mask_boundary(np.transpose(mask[:, :, ele_c]))  # same contour everywhere
roi_top_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).min() * sy
roi_bot_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).max() * sy

mc = seg_data.motion_compensation
ext_axial = [0, nx * sx, ny * sy, 0]

ticks = [0, 20, 40, 60, 80]

# Compute baseline noise floor from the first frame of the CEUS scan
first_frame = image_data.pixel_data[:, :, :, 0]*seg_data.seg_mask
noise_floor = compute_ceus_noise_floor(first_frame)*1.5

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for col, frame in enumerate(show_frames):
    Vol = image_data.pixel_data[:, :, :, frame]

    # --- row 0: NON-COMPENSATED (raw volume) ---
    raw = np.transpose(Vol[:, :, ele_c]).astype(np.float64)
    img = enhance_ceus(raw, p_low=noise_floor, p_high_percentile=99.5)
    axes[0, col].imshow(img, cmap='gray', extent=ext_axial, aspect='equal')
    axes[0, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[0, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

    # --- row 1: COMPENSATED (volume registered back) ---
    dx, dy, dz = mc.get_translation(frame)
    Vol_mc = ndshift(Vol, shift=[-dx, -dy, -dz], order=1, cval=0)
    raw_mc = np.transpose(Vol_mc[:, :, ele_c]).astype(np.float64)
    img_mc = enhance_ceus(raw_mc, p_low=noise_floor, p_high_percentile=99.5)
    axes[1, col].imshow(img_mc, cmap='gray', extent=ext_axial, aspect='equal')
    axes[1, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[1, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

# Apply tick marks to all axes
for ax in axes.flat:
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis='both', labelsize=16)
    

# Only show tick labels on the bottom row (x) and left column (y)
for col in range(5):
    axes[0, col].set_xticklabels([])          # hide x labels on top row
    if col != 0:
        axes[0, col].set_yticklabels([])      # hide y labels except leftmost
        axes[1, col].set_yticklabels([])

plt.tight_layout()
plt.show()

# Figure 4: Parameteric Map analysis

This script only works once finish the parametric analysis get the paramaps_LOGNORMAL. NPY files for different parameters

Figure 4: Parametric maps of 3D DCE analysis using motion compensation and without using motion compensation

In [ ]:
# ============================================================================
# MATPLOTLIB 2D VISUALIZATION
# ============================================================================
from typing import Optional, List
import os
import numpy as np
from matplotlib import pyplot as plt

def plot_heatmap_2d(
    param_map: np.ndarray,
    image_data=None,
    time_point: Optional[int] = None,
    seg_mask: Optional[np.ndarray] = None,
    title_name: str = 'T0 Map',
    colormap: str = 'turbo',
    param_opacity: float = 0.7,
    value_range: tuple = (0,100),
    pixel_size=None,
    transpose: bool = False,
    ax: Optional[plt.Axes] = None,
):
    """
    Display a single 2D parametric heatmap slice (e.g. a T0 or AUC plane) overlaid on
    a CEUS background image using matplotlib. Call this once per plane (axial, sagittal,
    coronal, ...) with the already-sliced 2D arrays for that plane.

    Args:
        param_map: 2D array of parametric values for this plane's slice
        image_data: Optional background image for this plane. Pass a 2D array already
            sliced to match param_map, or a 3D (rows, cols, T) array to pick a frame
            from via `time_point`.
        time_point: Which time frame of image_data to use as background, if image_data
            has a trailing time axis (None = middle frame)
        seg_mask: Optional 2D boolean/int mask (same shape as param_map) to outline the VOI
        title_name: Plot title / colorbar label
        colormap: Matplotlib colormap name for the parametric overlay
        param_opacity: Opacity of the parametric overlay
        range: Tuple of (min, max) values for the color scale limits
        pixel_size: Optional (row_spacing, col_spacing) voxel spacing in mm for this
            plane, used to render with a physically correct aspect ratio. Defaults to (1, 1).
        transpose: If True, transpose param_map/image_data/seg_mask before displaying
            (e.g. the axial plane from a (X, Y, Z) volume is stored as (X, Y) and needs
            transposing to display in (row, col) = (Y, X) order)
        ax: Optional existing matplotlib Axes to draw on; if None, a new figure is created

    Returns:
        fig, ax: The matplotlib Figure and Axes objects
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    else:
        fig = ax.figure

    def orient(arr):
        return np.transpose(arr) if transpose else arr

    row_spacing, col_spacing = pixel_size if pixel_size is not None else (1.0, 1.0)
    display_rows, display_cols = orient(param_map).shape
    extent = (0, display_cols * col_spacing, display_rows * row_spacing, 0)

    # ------------------------------------------------------------------
    # Background CEUS frame (optional)
    # ------------------------------------------------------------------
    if image_data is not None:
        background_slice = image_data
        if background_slice.ndim >= 3:
            if time_point is None:
                time_point = background_slice.shape[-1] // 2
            background_slice = background_slice[..., time_point]
        ax.imshow(orient(background_slice), cmap='gray', extent=extent, aspect='equal')

    # ------------------------------------------------------------------
    # Optional segmentation outline
    # ------------------------------------------------------------------
    if seg_mask is not None:
        mask_to_show = (orient(seg_mask) > 0).astype(float)
        ax.contour(mask_to_show, levels=[0.5], colors='white', linewidths=1, extent=extent)

    # ------------------------------------------------------------------
    # Parametric heatmap overlay
    # ------------------------------------------------------------------
    param_display = orient(param_map).astype(np.float64)
    param_masked = np.ma.masked_where(param_display <= 0, param_display)

    valid = param_display[param_display > 0]
    if len(valid) > 0:
        im = ax.imshow(
            param_masked,
            cmap=colormap,
            vmin=value_range[0],
            vmax=value_range[1],
            alpha=param_opacity,
            extent=extent,
            aspect='equal',
        )
        fig.colorbar(im, ax=ax, label=title_name)
        print(f"{title_name}: contrast=[{value_range[0]:.1f}, {value_range[1]:.1f}], "
              f"mean(activated)={np.mean(valid):.1f}, "
              f"activated voxels={len(valid)}")
    else:
        print(f"Warning: {title_name} has no activated voxels to display.")

    ax.set_title(title_name)
    ax.axis('off')

    return fig, ax

def orient_camera(p, grid, zoom=1.0):
    """Set X-horizontal, Y-vertical, Z-into-plane orientation.
    zoom < 1.0 zooms out, > 1.0 zooms in."""
    cx, cy, cz = grid.center
    xmin, xmax, ymin, ymax, zmin, zmax = grid.bounds
    span = max(xmax - xmin, ymax - ymin)
    dist = span * 2.5 / zoom          # smaller zoom -> larger dist -> further out
    p.camera_position = [
        (cx, cy, zmax - dist),
        (cx, cy, cz),
        (0.0, -1.0, 0.0),
    ]
    p.reset_camera_clipping_range()
    p.camera.zoom(zoom)               # also scales the view


def save_sweep_video(p, grid, out_dir, filename="auc_sweep.mp4",
                     total_deg=180.0, n_frames=90, framerate=20):
    """Rotate the camera through `total_deg` in azimuth and write a movie."""
    orient_camera(p, grid)
    step = total_deg / n_frames

    path = os.path.join(out_dir, filename)
    p.open_movie(path, framerate=framerate)
    p.write_frame()                       # first frame at starting orientation
    for _ in range(n_frames):
        p.camera.azimuth += step          # rotate around view-up (Y) axis
        p.reset_camera_clipping_range()
        p.write_frame()
    print(f"Saved video: {path}")
    return path


def save_sweep_frames(p, grid, out_dir, prefix="auc_frame",
                      total_deg=180.0, n_frames=90, save_every=18, scale=2,zoom=1.0):
    """Rotate through `total_deg` and save a screenshot every `save_every` frames."""
    orient_camera(p, grid,zoom=zoom)
    step = total_deg / n_frames

    saved = []
    for i in range(n_frames + 1):         # include frame 0
        if i > 0:
            p.camera.azimuth += step
            p.reset_camera_clipping_range()
        if i % save_every == 0:
            path = os.path.join(out_dir, f"{prefix}_{i:03d}.png")
            p.screenshot(path, scale=scale)
            saved.append(path)
            print(f"Saved frame: {path}")
    return saved

In [ ]:
# Loading the visualization path
visualization_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_Aceelerate'
AUC = np.load(os.path.join(visualization_path, 'TP_full_TIC_numerical.npy'))
image = np.load(os.path.join(visualization_path, 'image.npy'))
pixel_size = np.load(os.path.join(visualization_path, 'pix_dims.npy'))  # (dx, dy, dz) mm

x, y, z = AUC.shape[0] // 2, AUC.shape[1] // 2, AUC.shape[2] // 2

fig1, ax1 = plot_heatmap_2d(AUC[:, :, z], image_data=image[:, :, z],time_point = 15,param_opacity=0.4, transpose=True,value_range = (0,30), title_name='Axial')
fig2, ax2 = plot_heatmap_2d(AUC[x, :, :], image_data=image[x, :, :], time_point = 15,param_opacity=0.4, value_range = (0,30), title_name='Sagittal')
fig3, ax3 = plot_heatmap_2d(AUC[:, y, :], image_data=image[:, y, :], time_point = 15,param_opacity=0.4, value_range = (0,30), title_name='Coronal')

plt.show()

In [ ]:
visualization_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_mc'
AUC = np.load(os.path.join(visualization_path, 'TP_full_TIC_numerical.npy'))
image = np.load(os.path.join(visualization_path, 'image.npy'))
pixel_size = np.load(os.path.join(visualization_path, 'pix_dims.npy'))  # (dx, dy, dz) mm

x, y, z = AUC.shape[0] // 2, AUC.shape[1] // 2, AUC.shape[2] // 2

fig1, ax1 = plot_heatmap_2d(AUC[:, :, z], image_data=image[:, :, z],time_point = 15,param_opacity=0.4, transpose=True,value_range = (0,30), title_name='Axial')
fig2, ax2 = plot_heatmap_2d(AUC[x, :, :], image_data=image[x, :, :], time_point = 15,param_opacity=0.4, value_range = (0,30), title_name='Sagittal')
fig3, ax3 = plot_heatmap_2d(AUC[:, y, :], image_data=image[:, y, :], time_point = 15,param_opacity=0.4, value_range = (0,30), title_name='Coronal')


plt.show()

In [ ]:
import numpy as np
import pyvista as pv
from skimage.morphology import remove_small_objects
from skimage.morphology import binary_erosion, ball

# ============================================================================
# 1. HEADLESS / SSH SETUP  — must come before any plotting
# ============================================================================
pv.OFF_SCREEN = False                                 # no physical monitor on remote
pv.set_jupyter_backend('trame')
pv.global_theme.trame.server_proxy_enabled = False
pv.global_theme.font.color = 'white'                 # white text on black background

# ============================================================================
# 2. LOAD DATA
# ============================================================================
visualization_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_noMC'
AUC        = np.load(f'{visualization_path}/TP_full_TIC_numerical.npy')   # (X, Y, Z)
image      = np.load(f'{visualization_path}/image.npy')                   # (X, Y, Z, T) or (X, Y, Z)
pixel_size = np.load(f'{visualization_path}/pix_dims.npy')                # (dx, dy, dz) mm
dx, dy, dz = pixel_size

# ============================================================================
# 3. BUILD THE PARAMETRIC VOLUME
# ============================================================================
auc = AUC.astype(np.float32).copy()
auc[auc <= 0] = 0                        # non-activated -> 0 so they render transparent

# Sanity check — if this prints 0 activated voxels, the cloud will be blank
print(f"AUC range: [{np.nanmin(auc):.2f}, {np.nanmax(auc):.2f}], "
      f"activated voxels: {(auc > 0).sum()}")

grid = pv.ImageData()
grid.dimensions = np.array(auc.shape)               # POINT data -> dims match array shape
grid.spacing    = (float(dx), float(dy), float(dz)) # physically correct aspect ratio
grid.point_data["AUC"] = auc.flatten(order="F")     # VTK expects Fortran order

# ============================================================================
# 5. RENDER
# ============================================================================
opacity_tf = opacity_tf = [0.0, 0.0, 0.0, 0.05, 0.1, 0.4]     # transparent low end -> opaque high end

p = pv.Plotter()
p.set_background("white")

# bounding box: always visible, confirms the scene mounted even if the cloud is faint
bg = image[..., 20] if image.ndim == 4 else image
bg = bg.astype(np.float32)

bg_small_thresh = 70
mask = bg>bg_small_thresh
mask = binary_erosion(mask, ball(1))              # pure erosion — sharper, smaller
mask = remove_small_objects(mask, min_size=2000)

bg_clean = np.where(mask, bg, 0).astype(np.float32)  # remove small speckles
bg = bg_clean

bg_grid = pv.ImageData()
bg_grid.dimensions = np.array(bg.shape)
bg_grid.spacing    = (float(dx), float(dy), float(dz))
bg_grid.point_data["bg"] = bg.flatten(order="F")


# --- threshold: everything below `bg_thresh` is fully transparent ---
bg_thresh = 70        # raise to hide more dark pixels, lower to keep more tissue
bg_max    = 255

# opacity curve sampled evenly across clim (0..bg_max):
# stays 0 until the threshold, then ramps up
n = 5
levels = np.linspace(0, bg_max, n)
bg_opacity = np.where(levels < bg_thresh, 0.0,
                      np.linspace(0, 0.6, n))   # gentle ramp above threshold

p.add_volume(
    bg_grid, scalars="bg", cmap="gray",
    clim=(0, bg_max),
    opacity=bg_opacity,
    opacity_unit_distance=float(np.mean([dx, dy, dz])*4),
    shade=False,
    show_scalar_bar=False
)

p.add_volume(
    grid,
    scalars="AUC",
    cmap="jet",
    clim=(0, 30),
    opacity=opacity_tf,
    opacity_unit_distance=float(np.mean([dx, dy, dz])),
    shade=False,
    show_scalar_bar=False
)


# --- 180 degree azimuth sweep, written off-screen ---
out_dir = visualization_path  # or wherever you want output
n_frames = 30                 # 90 frames over 180 deg = 2 deg/frame
step = 180.0 / n_frames
saving_interval = 3

p.show_grid(color='black')

save_sweep_frames(p, grid, out_dir, prefix="auc_frame", total_deg=180.0, n_frames=n_frames, save_every=saving_interval, scale=2,zoom=0.9)

p.close()
